### ЗАДАЧА: Панель обработки возвратов и refund-выплат по паттерну `MVC`

Команда customer operations разбирает кейсы по возвратам заказов и согласованию refund-выплат.
Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `order_id` — номер заказа;
- `seller` — продавец;
- `items_amount` — стоимость товаров к возврату;
- `shipping_refund` — сумма возврата за доставку;
- `restocking_fee` — удержание за возврат/обработку;
- `requested_total` — общая сумма запроса на возврат;
- `approved_total` — одобренная сумма;
- `refund_gap` — разница между запросом и одобренной суммой;
- `status` — текущий статус;
- `reviewer` — сотрудник, который ведет кейс;
- `evidence_ready` — собраны ли материалы по кейсу;
- `decision` — итоговое решение.

### Формулы

При создании кейса и при изменении approved-суммы нужно правильно считать:
- `requested_total = items_amount + shipping_refund - restocking_fee`
- `refund_gap = requested_total - approved_total`
- оба значения нужно округлять до 2 знаков.

Также нужна сводка:
- количество кейсов по статусам;
- `total_requested` — общая сумма запросов;
- `total_approved` — общая одобренная сумма;
- `total_gap` — общая сумма `refund_gap`;
- `paid_total` — общая сумма по кейсам, завершенным как `paid`.

### Статусы

- `new`
- `reviewing`
- `evidence_requested`
- `evidence_received`
- `ready_for_payout`
- `paid`
- `rejected`
- `escalated`

### Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `reviewer` несуществующему кейсу;
- финальные кейсы (`paid`, `rejected`, `escalated`) нельзя менять дальше;
- начать review можно только из `new` и только если назначен `reviewer`;
- запросить доказательства можно только из `reviewing`;
- подтвердить получение доказательств можно только из `evidence_requested`;
- при подтверждении доказательств `evidence_ready` должно стать `True`, а статус — `evidence_received`;
- одобренную сумму можно устанавливать только из `reviewing` или `evidence_received`;
- `approved_total` не может быть меньше 0 и больше `requested_total`;
- после изменения `approved_total` нужно пересчитать `refund_gap`;
- перевод в `ready_for_payout` возможен только из `reviewing` или `evidence_received`;
- перевод в `ready_for_payout` возможен только если `approved_total > 0`;
- перевести кейс в `paid` можно только из `ready_for_payout`;
- отклонить кейс можно только из `reviewing`, `evidence_requested` или `evidence_received`;
- эскалировать кейс можно только из `reviewing`, `evidence_requested`, `evidence_received` или `ready_for_payout`.

In [3]:
from dataclasses import dataclass


initial_cases = [
    ("RR-100", "ORD-7001", "best-gadgets", 3200.0, 300.0, 150.0),
    ("RR-101", "ORD-7002", "home-decor-plus", 1800.0, 0.0, 100.0),
]

actions = [
    ("show",),
    ("review", "RR-100"),
    ("assign", "RR-100", "Olga"),
    ("review", "RR-100"),
    ("evidence_request", "RR-100"),
    ("evidence_receive", "RR-100"),
    ("approve_amount", "RR-100", 3200.0),
    ("payout_ready", "RR-100"),
    ("paid", "RR-100", "refund_sent"),
    ("create", "RR-102", "ORD-7003", "sport-zone", 2500.0, 200.0, 50.0),
    ("assign", "RR-102", "Max"),
    ("review", "RR-102"),
    ("approve_amount", "RR-102", 3000.0),
    ("escalate", "RR-102"),
    ("show",),
]


@dataclass
class RefundCase:
    case_id: str
    order_id: str
    seller: str
    items_amount: float
    shipping_refund: float
    restocking_fee: float
    requested_total: float
    approved_total: float
    refund_gap: float
    status: str = "new"
    reviewer: str | None = None
    evidence_ready: bool = False
    decision: str | None = None


class RefundModel:
    def __init__(self):
        self.cases = {}
        self.done_statuses = {"paid", "rejected","escalated"}

    def _calculate_requested_total(self, items_amount: float, shipping_refund: float, restocking_fee: float) -> float:
        return round(items_amount + shipping_refund - restocking_fee, 2)

    def _calculate_refund_gap(self, requested_total: float, approved_total: float) -> float:
        return round(requested_total - approved_total, 2)

    def add_case(self, case_id: str, order_id: str, seller: str, items_amount: float, shipping_refund: float, restocking_fee: float) -> None:
        if case_id in self.cases:
            raise ValueError("Case ID already exists")
        requested_total = self._calculate_requested_total(items_amount, shipping_refund, restocking_fee)
        refund_gap = self._calculate_refund_gap(requested_total, 0.0)
        self.cases[case_id] = RefundCase(case_id, order_id, seller, items_amount, shipping_refund, restocking_fee, requested_total, 0.0, refund_gap)

    def assign_reviewer(self, case_id: str, reviewer: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError("already done")
        self.cases[case_id].reviewer = reviewer

    def start_review(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError("already done")
        if self.cases[case_id].reviewer is None:
            raise ValueError("Reviewer not assigned")
        if self.cases[case_id].status != "new":
            raise ValueError("Case not in new status")
        
        self.cases[case_id].status = "reviewing"

    def request_evidence(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError("already done")
        if self.cases[case_id].status != "reviewing":
            raise ValueError("Case not in reviewing status")
        
        self.cases[case_id].status = "evidence_requested"

    def receive_evidence(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError("already done")
        if self.cases[case_id].status != "evidence_requested":
            raise ValueError("Case not in evidence_requested status")
        
        case = self.cases[case_id]
        case.evidence_ready = True
        case.status = "evidence_received"

    def set_approved_total(self, case_id: str, approved_total: float) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError("already done")
        if self.cases[case_id].status not in {"reviewing", "evidence_received"}:
            raise ValueError("Case not in reviewing or evidence_received status")
        if approved_total < 0 or approved_total > self.cases[case_id].requested_total:
            raise ValueError("Approved total must be between 0 and requested total")

        case = self.cases[case_id]
        case.approved_total = approved_total
        case.refund_gap = self._calculate_refund_gap(case.requested_total, approved_total)

    def mark_ready_for_payout(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError("already done")
        if self.cases[case_id].status not in {"evidence_received", "reviewing"}:
            raise ValueError("Case not in evidence_received or reviewing status")
        if self.cases[case_id].approved_total <= 0:
            raise ValueError("Approved total must be greater than 0 to mark ready for payout")
        
        self.cases[case_id].status = "ready_for_payout"

    def mark_paid(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError("already done")
        if self.cases[case_id].status != "ready_for_payout":
            raise ValueError("Case not in ready_for_payout status") 
        
        
        case = self.cases[case_id]
        case.status = "paid"
        case.decision = decision

    def reject_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError("already done")
        if self.cases[case_id].status not in {"reviewing", "evidence_received", "ready_for_payout"}:
            raise ValueError("Case not in reviewing, evidence_received or ready_for_payout status")
        
        case = self.cases[case_id]
        case.status = "rejected"
        case.decision = decision

    def escalate_case(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError("already done")
        if self.cases[case_id].status not in {"reviewing", "evidence_received", "ready_for_payout","evidence_requested"}:
            raise ValueError("Case not in reviewing, evidence_received or ready_for_payout status or evidence_requested")
        
        self.cases[case_id].status = "escalated"

    def list_cases(self) -> list[str]:
        rows = []
        for case in self.cases.values():
            rows.append(
                f"{case.case_id} | {case.order_id} | {case.seller} | {case.items_amount} | {case.shipping_refund} | {case.restocking_fee} | "
                f"{case.requested_total} | {case.approved_total} | {case.refund_gap} | {case.status} | {case.reviewer} | {case.evidence_ready} | {case.decision}"
            )
        return rows

    def summary(self) -> dict[str, float | int]:
        result = {
            "total_cases": 0,
            "total_requested": 0.0,
            "total_approved": 0.0,
            "total_gap": 0.0,
            "paid_total": 0.0,
        }
        for case in self.cases.values():
            result[case.status] = result.get(case.status, 0) + 1
            result["total_cases"] += 1
            result["total_requested"] += case.approved_total
            result["total_approved"] += case.requested_total
            result["total_gap"] += abs(case.approved_total)
            if case.status == "paid":
                result["paid_total"] += case.requested_total
        return result


class RefundView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("Refund cases:")
        for row in rows:
            print(row)

    @staticmethod
    def render_summary(summary: dict[str, float | int]) -> None:
        print("Summary:", summary)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)


class RefundController:
    def __init__(self, model: RefundModel, view: RefundView):
        self.model = model
        self.view = view

    def create_case(self, case_id: str, order_id: str, seller: str, items_amount: float, shipping_refund: float, restocking_fee: float) -> None:
        try:
            self.model.add_case(case_id, order_id, seller, items_amount, shipping_refund, restocking_fee)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_reviewer(self, case_id: str, reviewer: str) -> None:
        try:
            self.model.assign_reviewer(case_id, reviewer)
            self.view.render_success(f"Reviewer assigned to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_review(self, case_id: str) -> None:
        try:
            self.model.start_review(case_id)
            self.view.render_success(f"Case {case_id} moved to review")
        except ValueError as error:
            self.view.render_error(str(error))

    def request_evidence(self, case_id: str) -> None:
        try:
            self.model.request_evidence(case_id)
            self.view.render_success(f"Evidence requested for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def receive_evidence(self, case_id: str) -> None:
        try:
            self.model.receive_evidence(case_id)
            self.view.render_success(f"Evidence received for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def set_approved_total(self, case_id: str, approved_total: float) -> None:
        try:
            self.model.set_approved_total(case_id, approved_total)
            self.view.render_success(f"Approved total set for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def mark_ready_for_payout(self, case_id: str) -> None:
        try:
            self.model.mark_ready_for_payout(case_id)
            self.view.render_success(f"Case {case_id} is ready for payout")
        except ValueError as error:
            self.view.render_error(str(error))

    def mark_paid(self, case_id: str, decision: str) -> None:
        try:
            self.model.mark_paid(case_id, decision)
            self.view.render_success(f"Case {case_id} marked as paid")
        except ValueError as error:
            self.view.render_error(str(error))

    def reject_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.reject_case(case_id, decision)
            self.view.render_success(f"Case {case_id} rejected")
        except ValueError as error:
            self.view.render_error(str(error))

    def escalate_case(self, case_id: str) -> None:
        try:
            self.model.escalate_case(case_id)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())
        self.view.render_summary(self.model.summary())


model = RefundModel()
view = RefundView()
controller = RefundController(model, view)

for case_id, order_id, seller, items_amount, shipping_refund, restocking_fee in initial_cases:
    model.add_case(case_id, order_id, seller, items_amount, shipping_refund, restocking_fee)

for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, order_id, seller, items_amount, shipping_refund, restocking_fee = action
        controller.create_case(case_id, order_id, seller, items_amount, shipping_refund, restocking_fee)
    elif action[0] == "assign":
        _, case_id, reviewer = action
        controller.assign_reviewer(case_id, reviewer)
    elif action[0] == "review":
        _, case_id = action
        controller.start_review(case_id)
    elif action[0] == "evidence_request":
        _, case_id = action
        controller.request_evidence(case_id)
    elif action[0] == "evidence_receive":
        _, case_id = action
        controller.receive_evidence(case_id)
    elif action[0] == "approve_amount":
        _, case_id, approved_total = action
        controller.set_approved_total(case_id, approved_total)
    elif action[0] == "payout_ready":
        _, case_id = action
        controller.mark_ready_for_payout(case_id)
    elif action[0] == "paid":
        _, case_id, decision = action
        controller.mark_paid(case_id, decision)
    elif action[0] == "reject":
        _, case_id, decision = action
        controller.reject_case(case_id, decision)
    elif action[0] == "escalate":
        _, case_id = action
        controller.escalate_case(case_id)

print()
print("Финальное состояние:")
controller.show_cases()

Refund cases:
RR-100 | ORD-7001 | best-gadgets | 3200.0 | 300.0 | 150.0 | 3350.0 | 0.0 | 3350.0 | new | None | False | None
RR-101 | ORD-7002 | home-decor-plus | 1800.0 | 0.0 | 100.0 | 1700.0 | 0.0 | 1700.0 | new | None | False | None
Summary: {'total_cases': 2, 'total_requested': 0.0, 'total_approved': 5050.0, 'total_gap': 0.0, 'paid_total': 0.0, 'new': 2}
ERROR: Reviewer not assigned
SUCCESS: Reviewer assigned to RR-100
SUCCESS: Case RR-100 moved to review
SUCCESS: Evidence requested for RR-100
SUCCESS: Evidence received for RR-100
SUCCESS: Approved total set for RR-100
SUCCESS: Case RR-100 is ready for payout
SUCCESS: Case RR-100 marked as paid
SUCCESS: Case RR-102 created
SUCCESS: Reviewer assigned to RR-102
SUCCESS: Case RR-102 moved to review
ERROR: Approved total must be between 0 and requested total
SUCCESS: Case RR-102 escalated
Refund cases:
RR-100 | ORD-7001 | best-gadgets | 3200.0 | 300.0 | 150.0 | 3350.0 | 3200.0 | 150.0 | paid | Olga | True | refund_sent
RR-101 | ORD-7002

In [1]:
# Refund cases:
# RR-100 | ORD-7001 | best-gadgets | 3200.0 | 300.0 | 150.0 | 3350.0 | 0.0 | 3350.0 | new | None | False | None
# RR-101 | ORD-7002 | home-decor-plus | 1800.0 | 0.0 | 100.0 | 1700.0 | 0.0 | 1700.0 | new | None | False | None
# Summary: {'total_cases': 2, 'total_requested': 5050.0, 'total_approved': 0.0, 'total_gap': 5050.0, 'paid_total': 0.0, 'new': 2}
# ERROR: Reviewer is required
# SUCCESS: Reviewer assigned to RR-100
# SUCCESS: Case RR-100 moved to review
# SUCCESS: Evidence requested for RR-100
# SUCCESS: Evidence received for RR-100
# SUCCESS: Approved total set for RR-100
# SUCCESS: Case RR-100 is ready for payout
# SUCCESS: Case RR-100 marked as paid
# SUCCESS: Case RR-102 created
# SUCCESS: Reviewer assigned to RR-102
# SUCCESS: Case RR-102 moved to review
# ERROR: Approved total is too large
# SUCCESS: Case RR-102 escalated
# Refund cases:
# RR-100 | ORD-7001 | best-gadgets | 3200.0 | 300.0 | 150.0 | 3350.0 | 3200.0 | 150.0 | paid | Olga | True | refund_sent
# RR-101 | ORD-7002 | home-decor-plus | 1800.0 | 0.0 | 100.0 | 1700.0 | 0.0 | 1700.0 | new | None | False | None
# RR-102 | ORD-7003 | sport-zone | 2500.0 | 200.0 | 50.0 | 2650.0 | 0.0 | 2650.0 | escalated | Max | False | escalated
# Summary: {'total_cases': 3, 'total_requested': 7700.0, 'total_approved': 3200.0, 'total_gap': 4500.0, 'paid_total': 3200.0, 'paid': 1, 'new': 1, 'escalated': 1}
#
# Финальное состояние:
# Refund cases:
# RR-100 | ORD-7001 | best-gadgets | 3200.0 | 300.0 | 150.0 | 3350.0 | 3200.0 | 150.0 | paid | Olga | True | refund_sent
# RR-101 | ORD-7002 | home-decor-plus | 1800.0 | 0.0 | 100.0 | 1700.0 | 0.0 | 1700.0 | new | None | False | None
# RR-102 | ORD-7003 | sport-zone | 2500.0 | 200.0 | 50.0 | 2650.0 | 0.0 | 2650.0 | escalated | Max | False | escalated
# Summary: {'total_cases': 3, 'total_requested': 7700.0, 'total_approved': 3200.0, 'total_gap': 4500.0, 'paid_total': 3200.0, 'paid': 1, 'new': 1, 'escalated': 1}